# EWL Mechanism Analysis — Velocity × EMA Scatter

Visualises per-sample EWL behaviour across datasets.

**Run first:**
```bash
python scripts/run_diagnostic_scatter.py --gpus 0 1 2 --mode noise
```

Fig A — Velocity vs EMA scatter (coloured noisy/clean, sized by weight)  
Fig B — Weight KDE: noisy vs clean samples  
Fig C — Temporal: mean weight of noisy vs clean over epochs

In [1]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import gaussian_kde

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "legend.framealpha": 0.9,
    "legend.fontsize": 9,
})

C_NOISY = "#D6604D"   # muted red
C_CLEAN = "#2166AC"   # steel blue

BASE_DIR = Path("../outputs/vision/diagnostic")

DATASETS = {
    "aircraft":      "FGVC-Aircraft",
    "cub200":        "CUB-200",
    "stanford_dogs": "Stanford Dogs",
}
CONDITION = "ewl_noise20pct"   # change to ewl_imbalance10pct for imbalance mode

print("Config ready.")

Config ready.


In [3]:
def load_epoch(dataset, condition, epoch):
    """Load sample_stats_ep{epoch:02d}.npz for a given dataset/condition."""
    path = BASE_DIR / dataset / condition / "sample_stats" / f"sample_stats_ep{epoch:02d}.npz"
    if not path.exists():
        return None
    d = np.load(path)
    return {k: d[k] for k in d.files}


def load_noisy_mask(dataset, condition):
    path = BASE_DIR / dataset / condition / "sample_stats" / "noisy_mask.npy"
    if not path.exists():
        return None
    return np.load(path)


def available_epochs(dataset, condition):
    d = BASE_DIR / dataset / condition / "sample_stats"
    if not d.exists():
        return []
    return sorted(int(p.stem.split("ep")[1]) for p in d.glob("sample_stats_ep*.npz"))


# Quick sanity check
for ds in DATASETS:
    eps = available_epochs(ds, CONDITION)
    mask = load_noisy_mask(ds, CONDITION)
    if mask is not None:
        n_noisy = mask.sum()
        n_total = len(mask)
        print(f"{ds:15s}: {len(eps)} epochs saved  |  "
              f"noisy={n_noisy}/{n_total} ({100*n_noisy/n_total:.1f}%)")
    else:
        print(f"{ds:15s}: {len(eps)} epochs saved  |  no noisy_mask found")

aircraft       : 9 epochs saved  |  noisy=637/3334 (19.1%)
cub200         : 9 epochs saved  |  noisy=925/4800 (19.3%)
stanford_dogs  : 10 epochs saved  |  noisy=1871/9600 (19.5%)


## Figure A — Velocity vs EMA scatter

Each point is one training sample at a given epoch.  
- **X-axis**: `curr_ema` — persistent loss level  
- **Y-axis**: `velocity` — relative per-step progress (+ve = improving)  
- **Colour**: noisy (red) vs clean (blue)  
- **Size/alpha**: EWL weight assigned to the sample

In [4]:
SCATTER_EPOCHS = [2, 3, 4, 10]   # which epochs to show as columns
MAX_POINTS = 800              # downsample per class to avoid overplotting

datasets = [ds for ds in DATASETS if available_epochs(ds, CONDITION)]
n_ds  = len(datasets)
n_ep  = len(SCATTER_EPOCHS)

fig, axes = plt.subplots(n_ds, n_ep, figsize=(3.8 * n_ep, 3.5 * n_ds), squeeze=False)

for row, ds in enumerate(datasets):
    noisy_mask = load_noisy_mask(ds, CONDITION)
    eps_avail  = available_epochs(ds, CONDITION)

    for col, ep in enumerate(SCATTER_EPOCHS):
        ax = axes[row][col]
        ep_use = min(ep, max(eps_avail)) if eps_avail else ep
        data = load_epoch(ds, CONDITION, ep_use)
        if data is None:
            ax.set_visible(False)
            continue

        ids      = data["ids"]
        curr_ema = data["curr_ema"]
        velocity = data["velocity"]
        weight   = data["weight"]

        # Build noisy flag for this batch (sample ids → noisy_mask)
        if noisy_mask is not None and len(noisy_mask) > ids.max():
            is_noisy = noisy_mask[ids]
        else:
            is_noisy = np.zeros(len(ids), dtype=bool)

        # Normalise weights to [0,1] for alpha/size
        w_norm = (weight - weight.min()) / (weight.max() - weight.min() + 1e-12)

        for flag, color, label in [
            (is_noisy,  C_NOISY, "Noisy"),
            (~is_noisy, C_CLEAN, "Clean"),
        ]:
            idx = np.where(flag)[0]
            if len(idx) == 0:
                continue
            # Downsample if needed
            if len(idx) > MAX_POINTS:
                rng = np.random.default_rng(0)
                idx = rng.choice(idx, MAX_POINTS, replace=False)

            ax.scatter(
                curr_ema[idx], velocity[idx],
                c=color,
                s=8 + 30 * w_norm[idx],
                alpha=np.clip(0.15 + 0.6 * w_norm[idx], 0.1, 0.8),
                linewidths=0,
                label=label,
                rasterized=True,
            )

        ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
        ax.set_xlabel("EMA loss (curr_ema)" if row == n_ds - 1 else "")
        ax.set_ylabel("Velocity" if col == 0 else "")
        ax.set_title(f"{DATASETS[ds]}  —  epoch {ep_use}")
        if row == 0 and col == n_ep - 1:
            ax.legend(markerscale=2)

        # Annotate quadrants
        xlim = ax.get_xlim(); ylim = ax.get_ylim()
        ax.text(0.97, 0.97, "Hard-clean\n(upweighted)",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=7, color=C_CLEAN, alpha=0.8)
        ax.text(0.97, 0.03, "Noisy\n(downweighted)",
                transform=ax.transAxes, ha="right", va="bottom",
                fontsize=7, color=C_NOISY, alpha=0.8)

plt.suptitle("Velocity vs EMA loss  (size/alpha ∝ EWL weight, noise=20%)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("fig_scatter_velocity_ema.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Figure B — Weight distribution: noisy vs clean (KDE)

Shows how well EWL separates noisy and clean samples by weight.
Well-separated = EWL is doing its job. Overlapping = signal is not discriminative.

In [5]:
KDE_EPOCH = 5   # which epoch to show

fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4), sharey=False)
if n_ds == 1: axes = [axes]

for ax, ds in zip(axes, datasets):
    noisy_mask = load_noisy_mask(ds, CONDITION)
    eps_avail  = available_epochs(ds, CONDITION)
    ep_use     = min(KDE_EPOCH, max(eps_avail)) if eps_avail else KDE_EPOCH
    data       = load_epoch(ds, CONDITION, ep_use)
    if data is None or noisy_mask is None:
        ax.set_visible(False)
        continue

    ids    = data["ids"]
    weight = data["weight"]
    is_noisy = noisy_mask[ids] if len(noisy_mask) > ids.max() else np.zeros(len(ids), dtype=bool)

    for flag, color, label in [(is_noisy, C_NOISY, "Noisy"), (~is_noisy, C_CLEAN, "Clean")]:
        w = weight[flag]
        if len(w) < 2: continue
        # KDE
        w_grid = np.linspace(weight.min(), weight.max(), 300)
        kde    = gaussian_kde(w, bw_method=0.2)
        ax.fill_between(w_grid, kde(w_grid), alpha=0.35, color=color)
        ax.plot(w_grid, kde(w_grid), color=color, lw=2, label=label)
        ax.axvline(np.mean(w), color=color, lw=1.2, ls="--", alpha=0.7)

    ax.set_xlabel("EWL weight")
    ax.set_ylabel("Density" if ds == datasets[0] else "")
    ax.set_title(f"{DATASETS[ds]}  (epoch {ep_use})")
    ax.legend()

plt.suptitle("EWL weight distribution: noisy vs clean samples  (noise=20%)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("fig_weight_kde_noisy_clean.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Figure C — Temporal evolution of mean weight: noisy vs clean

If hypothesis holds: on Aircraft, mean weight of noisy samples decreases over epochs relative to clean.  
On CUB/Dogs: the two lines should stay close or even cross — EWL can't separate them.

In [6]:
fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4), sharey=False)
if n_ds == 1: axes = [axes]

for ax, ds in zip(axes, datasets):
    noisy_mask = load_noisy_mask(ds, CONDITION)
    eps_avail  = available_epochs(ds, CONDITION)
    if not eps_avail or noisy_mask is None:
        ax.set_visible(False)
        continue

    mean_noisy, mean_clean = [], []
    std_noisy,  std_clean  = [], []

    for ep in eps_avail:
        data = load_epoch(ds, CONDITION, ep)
        if data is None: continue
        ids      = data["ids"]
        weight   = data["weight"]
        is_noisy = noisy_mask[ids] if len(noisy_mask) > ids.max() else np.zeros(len(ids), dtype=bool)

        w_noisy = weight[is_noisy]
        w_clean = weight[~is_noisy]

        mean_noisy.append(w_noisy.mean() if len(w_noisy) else np.nan)
        mean_clean.append(w_clean.mean() if len(w_clean) else np.nan)
        std_noisy.append(w_noisy.std()  if len(w_noisy) else np.nan)
        std_clean.append(w_clean.std()  if len(w_clean) else np.nan)

    eps  = np.array(eps_avail)
    mn   = np.array(mean_noisy); sn = np.array(std_noisy)
    mc   = np.array(mean_clean); sc = np.array(std_clean)

    ax.plot(eps, mn, color=C_NOISY, lw=2, label="Noisy")
    ax.fill_between(eps, mn - sn, mn + sn, color=C_NOISY, alpha=0.15)
    ax.plot(eps, mc, color=C_CLEAN, lw=2, label="Clean")
    ax.fill_between(eps, mc - sc, mc + sc, color=C_CLEAN, alpha=0.15)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Mean EWL weight" if ds == datasets[0] else "")
    ax.set_title(DATASETS[ds])
    ax.legend()

plt.suptitle("Mean EWL weight over epochs: noisy vs clean samples  (noise=20%)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("fig_weight_temporal.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Figure D — Separation score over epochs

Single scalar: `mean_weight(clean) - mean_weight(noisy)` per epoch per dataset.  
Positive = EWL correctly favours clean samples. Near-zero = no discrimination.

In [7]:
DS_COLORS = {
    "aircraft":      "#1A9641",
    "cub200":        "#F4A582",
    "stanford_dogs": "#7B2D8B",
}

fig, ax = plt.subplots(figsize=(7, 4))

for ds in datasets:
    noisy_mask = load_noisy_mask(ds, CONDITION)
    eps_avail  = available_epochs(ds, CONDITION)
    if not eps_avail or noisy_mask is None:
        continue

    sep = []
    for ep in eps_avail:
        data = load_epoch(ds, CONDITION, ep)
        if data is None: continue
        ids      = data["ids"]
        weight   = data["weight"]
        is_noisy = noisy_mask[ids] if len(noisy_mask) > ids.max() else np.zeros(len(ids), dtype=bool)
        w_noisy  = weight[is_noisy]
        w_clean  = weight[~is_noisy]
        sep.append(
            (w_clean.mean() if len(w_clean) else np.nan) -
            (w_noisy.mean() if len(w_noisy) else np.nan)
        )

    ax.plot(eps_avail, sep, lw=2.5, marker="o", markersize=4,
            color=DS_COLORS.get(ds, "gray"), label=DATASETS[ds])

ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("mean_weight(clean) − mean_weight(noisy)")
ax.set_title("EWL separation score: clean vs noisy  (noise=20%)")
ax.legend()

plt.tight_layout()
plt.savefig("fig_separation_score.pdf", bbox_inches="tight", dpi=300)
plt.show()